# Automated PDF Ingestion into Google Cloud Storage


This notebook outlines a robust workflow to transfer large volumes of PDF documents into a Google Cloud Storage (GCS) bucket leveraging the official Python client

## Agenda
1. **Prerequisites**: Environment setup and authentication
2. **Upload Function**: Defining a reusable GCS upload utility
3. **Batch Processing**: Iterating over PDF assets for bulk ingestion
4. **Validation & Logging**: Ensuring integrity and traceability of uploads


In [ ]:
import os
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = os.getenv("SA_ACCOUNT_CREDENTIALS")

In [ ]:
import os
import pandas as pd
import requests
from google.cloud import storage
from tqdm.notebook import tqdm

CSV_PATH = "./data/fallos_2024.csv"
BUCKET_NAME = "jurisprag-rawpdfs"

client = storage.Client()
print(client)
bucket = client.bucket(BUCKET_NAME)
print(bucket)

<Bucket: jurisprag-rawpdfs>


In [ ]:
def download_and_upload(url, bucket, destination_blob_name):
    """Descarga un PDF desde `url` y lo sube al bucket con el nombre `destination_blob_name`."""
    response = requests.get(url, stream=True)
    response.raise_for_status()
    blob = bucket.blob(destination_blob_name)
    blob.upload_from_string(response.content, content_type='application/pdf')


In [ ]:

df = pd.read_csv(CSV_PATH)
df = df[1532:]

assert 'DownloadURL' in df.columns, "La columna 'DownloadURL' no está en el CSV"

total_rows = len(df)

for idx, row in tqdm(df.iterrows(), total=total_rows, desc="Uploading pdfs to GCS"):
    pdf_url = row['DownloadURL']
    
    filename = os.path.basename(pdf_url.split('?')[0]) or f'document_{idx}.pdf'
    blob_name = f"pdfs/{filename}"
    
    download_and_upload(pdf_url, bucket, blob_name)
    

Uploading pdfs to GCS:  42%|████▏     | 6854/16223 [1:54:03<2:44:56,  1.06s/it]

```
(.venv) (base) ➜  jurisprudence-rag-ai git:(feature/dev_google_adk) ✗ python 02_process_pdfs.py

Procesando PDFs: 100%|██████████████████| 17755/17755 [4:03:55<00:00,  1.21it/s]

Proceso completado. Revisa los JSON en GCS.
```